In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CEBE2DFCB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CEBE524830>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CEBE2DFCB0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CEBE524830>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out o

In [7]:
##without structured output
model.invoke("Provide details about the moview Dune")


AIMessage(content="<think>\nOkay, so I need to provide details about the movie Dune. Let me start by recalling what I know. Dune is a science fiction film, right? It's based on a book, maybe Frank Herbert's novel? I think the first movie was made a while ago, like in 1984, and now there's a new version directed by Denis Villeneuve. There's also a sequel to the new movie, Dune Part Two. \n\nFirst, I should confirm the director and the year. The original 1984 one was directed by David Lynch. The newer one is definitely Denis Villeneuve, released in 2021, and the sequel in 2023. The source material is the Dune series by Frank Herbert. The first book is Dune, then Dune Messiah, and so on. The movie covers the first book, I believe, and the sequel will cover parts of the second book, Dune Messiah, and maybe some of the third, Children of Dune. \n\nThe main character is Paul Atreides, played by Timothée Chalamet in the new movies. The cast includes Rebecca Ferguson as Lady Jessica, Oscar Isa

In [8]:
response=model_with_structure.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [9]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me recall what I know about that movie. Inception is a 2010 film directed by Christopher Nolan. It\'s a sci-fi action movie that involves dreams within dreams. The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, and Ellen Page. The plot revolves around a thief who enters people\'s dreams to steal secrets, and he\'s given a final task of planting an idea into someone\'s mind, which is called "inception."\n\nNow, looking at the tools provided, there\'s a Movie function that requires title, year, director, and rating. The user didn\'t specify the year, but I know Inception was released in 2010. The director is Christopher Nolan. As for the rating, I think it\'s highly rated. On IMDb, it\'s around 8.8, and on Rotten Tomatoes, it\'s 88%. But the function probably expects a numerical rating out of 10. Let me check, yes, IMDb\'s rating is 8.8. 

In [10]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames')], genres=['Science Fiction', 'Action'], budget=160.0)

Structured output using TypedDic


In [12]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [13]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'}],
 'genres': ['Action', 'Sci-Fi', 'Adventure'],
 'title': 'Avengers',
 'year': 2012}